# Imports

In [39]:
import os 
import time
import h5py
import numpy as np
import pandas as pd
import joblib
from skimage import measure
from scipy.ndimage import zoom

# Params

In [20]:
sdf_dir = '/data/workspaces/spanwar/dataset/thingi/thingi_large_all/gt_thingi_large'
thingi_filenames_path = '/user/spanwar/home/Documents/learn-fvdb/ssu/SSU/run/thingi30.txt'
grid_sizes = [33, 65, 129]

# Utils

In [21]:
def load_3d_array(filename, input_dir, size):
    """
    Load a 3D array from a given file path.
    """
    # load h5py file
    file_path = os.path.join(input_dir, filename)
    with h5py.File(file_path, 'r') as f:
        data = f[f'{size-1}_sdf'][:]  
    return data

In [22]:
with open(thingi_filenames_path, 'r') as f:
    thingi_filenames = [line.strip() for line in f.readlines()]

In [28]:
def run_time_benchmark(logic_function, *args, **kwargs):
    execution_time_results = []
    for size in grid_sizes:
        for filename in thingi_filenames: 
            # Measure execution time
            start_time = time.time()
            logic_function(filename, size, *args, **kwargs)
            end_time = time.time()
            exec_time = end_time - start_time
            execution_time_results.append((filename, size, exec_time))
            # print(f'Filename: {filename}, Size: {size}, Execution Time: {exec_time:.6f} seconds')
    return execution_time_results

In [45]:
def run_time_benchmark_parallel(logic_function, *args, **kwargs):
    def excecute_wrapper(filename, size, *args, **kwargs):
        start_time = time.time()
        logic_function(filename, size, *args, **kwargs)
        end_time = time.time()
        exec_time = end_time - start_time
        return filename, size, exec_time
    
    execution_time_results = []
    for size in grid_sizes:
        out = joblib.Parallel(n_jobs=-1)(
            joblib.delayed(excecute_wrapper)(filename, size, *args, **kwargs) 
            for filename in thingi_filenames
        )  
        execution_time_results.extend(out)  
    return execution_time_results

In [56]:
def describe_exe_results(results):
    df = pd.DataFrame(results, columns=['Filename', 'Grid Size', 'Execution Time (s)'])
    summary = df.groupby('Grid Size')['Execution Time (s)'].agg(['mean', 'std', 'min', 'max']).reset_index()
    summary[['mean', 'std', 'min', 'max']] = summary[['mean', 'std', 'min', 'max']].applymap(lambda x: f"{x:.3f}")
    return summary

# Marching Cubes

In [57]:
def marching_cubes(filename, size, level=0.0):
    """
    Apply the Marching Cubes algorithm to extract a mesh from the SDF.
    """
    filename = filename + '.hdf5'
    sdf = load_3d_array(filename, sdf_dir, size)
    verts, faces, normals, values = measure.marching_cubes(sdf, level=level)

In [60]:
results = run_time_benchmark_parallel(marching_cubes)

In [62]:
describe_exe_results(results)

/tmp/ipykernel_2976598/1331356525.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  summary[['mean', 'std', 'min', 'max']] = summary[['mean', 'std', 'min', 'max']].applymap(lambda x: f"{x:.3f}")


,Grid Size,mean,std,min,max
0,33,0.019,0.003,0.014,0.024
1,65,0.031,0.005,0.015,0.040
2,129,0.136,0.022,0.105,0.167


In [32]:
describe_exe_results(results)

,Grid Size,mean,std,min,max
0,33,0.002914,0.001592,0.001982,0.009503
1,65,0.010767,0.000432,0.009927,0.011492
2,129,0.073171,0.002569,0.066523,0.078895


# Trilinear

In [63]:
def trilinear(filename, size):
    """
    Apply trilinear interpolation to resize the SDF.
    """
    filename = filename + '.hdf5'
    sdf = load_3d_array(filename, sdf_dir, size)
    up_size = (size-1)*4 + 1
    scale_factors = (up_size / size, up_size / size, up_size / size)
    resized_sdf = zoom(sdf, scale_factors, order=1, 
                       mode="nearest",     # boundary handling
                       prefilter=True )  
    print(f'filename: {filename}, resized shape: {resized_sdf.shape}')

In [64]:
results = run_time_benchmark_parallel(trilinear)

filename: 527631.hdf5, resized shape: (129, 129, 129)
filename: 79241.hdf5, resized shape: (129, 129, 129)
filename: 92763.hdf5, resized shape: (129, 129, 129)
filename: 316358.hdf5, resized shape: (129, 129, 129)
filename: 72960.hdf5, resized shape: (129, 129, 129)
filename: 313444.hdf5, resized shape: (129, 129, 129)
filename: 68381.hdf5, resized shape: (129, 129, 129)
filename: 72870.hdf5, resized shape: (129, 129, 129)
filename: 47984.hdf5, resized shape: (129, 129, 129)
filename: 75662.hdf5, resized shape: (129, 129, 129)
filename: 73075.hdf5, resized shape: (129, 129, 129)
filename: 77245.hdf5, resized shape: (129, 129, 129)
filename: 398259.hdf5, resized shape: (129, 129, 129)
filename: 78671.hdf5, resized shape: (129, 129, 129)
filename: 75496.hdf5, resized shape: (129, 129, 129)
filename: 75665.hdf5, resized shape: (129, 129, 129)
filename: 64764.hdf5, resized shape: (129, 129, 129)
filename: 75656.hdf5, resized shape: (129, 129, 129)
filename: 252119.hdf5, resized shape: (129

In [65]:
describe_exe_results(results)

/tmp/ipykernel_2976598/1331356525.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  summary[['mean', 'std', 'min', 'max']] = summary[['mean', 'std', 'min', 'max']].applymap(lambda x: f"{x:.3f}")


,Grid Size,mean,std,min,max
0,33,0.164,0.033,0.118,0.229
1,65,1.086,0.167,0.775,1.464
2,129,7.597,0.517,6.161,8.660


In [35]:
describe_exe_results(results)

,Grid Size,mean,std,min,max
0,33,0.092878,0.004208,0.090341,0.113244
1,65,0.725435,0.009013,0.715801,0.749222
2,129,5.709023,0.051413,5.643027,5.814410


# Bspline

In [66]:
def bspline(filename, size):
    """
    Apply B-spline interpolation to resize the SDF.
    """
    filename = filename + '.hdf5'
    sdf = load_3d_array(filename, sdf_dir, size)
    up_size = (size-1)*4 + 1
    scale_factors = (up_size / size, up_size / size, up_size / size)
    resized_sdf = zoom(sdf, scale_factors, order=3, 
                       mode="nearest",     # boundary handling
                       prefilter=True )  
    print(f'filename: {filename}, resized shape: {resized_sdf.shape}')

In [67]:
results = run_time_benchmark_parallel(bspline)

filename: 44234.hdf5, resized shape: (129, 129, 129)
filename: 316358.hdf5, resized shape: (129, 129, 129)
filename: 64764.hdf5, resized shape: (129, 129, 129)
filename: 47984.hdf5, resized shape: (129, 129, 129)
filename: 75656.hdf5, resized shape: (129, 129, 129)
filename: 92763.hdf5, resized shape: (129, 129, 129)
filename: 441708.hdf5, resized shape: (129, 129, 129)
filename: 73075.hdf5, resized shape: (129, 129, 129)
filename: 313444.hdf5, resized shape: (129, 129, 129)
filename: 76277.hdf5, resized shape: (129, 129, 129)
filename: 95444.hdf5, resized shape: (129, 129, 129)
filename: 72960.hdf5, resized shape: (129, 129, 129)
filename: 90889.hdf5, resized shape: (129, 129, 129)
filename: 53159.hdf5, resized shape: (129, 129, 129)
filename: 75665.hdf5, resized shape: (129, 129, 129)
filename: 75655.hdf5, resized shape: (129, 129, 129)
filename: 79241.hdf5, resized shape: (129, 129, 129)
filename: 75496.hdf5, resized shape: (129, 129, 129)
filename: 75662.hdf5, resized shape: (129, 

In [68]:
describe_exe_results(results)

/tmp/ipykernel_2976598/1331356525.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  summary[['mean', 'std', 'min', 'max']] = summary[['mean', 'std', 'min', 'max']].applymap(lambda x: f"{x:.3f}")


,Grid Size,mean,std,min,max
0,33,0.964,0.122,0.847,1.292
1,65,6.637,0.425,5.410,7.309
2,129,49.006,0.227,48.692,49.303


In [54]:
describe_exe_results(results)

,Grid Size,mean,std,min,max
0,33,0.907448,0.088136,0.668434,1.067411
1,65,6.477091,0.488608,5.295040,7.229679
2,129,48.743164,0.141935,48.454491,49.090468
